# Gemma4 backend matrix v3 — cold-cache legs + 50K depth

Continues from v2/phase-2. Established so far (A100 80G, vllm 0.28.0,
text-only serving, 30K prompt):

| leg | decode tok/s |
|---|---:|
| TRITON + MTP (default routing) | 31.50 |
| TRITON, no spec | 43.85 |
| explicit FLASHINFER + MTP | 62.43 |

and one root cause: the torch_aot_compile cache key includes an
explicitly set `attention_config.backend` but NOT the auto-resolved
backend nor `limit_mm_per_prompt`; runs 3b/4b loaded a stale artifact
compiled under TRITON+mm-on and crashed with
`AttributeError: 'NoneType' object has no attribute 'size'`.

**This notebook**: every leg clears the compile cache first (~2 min
extra per leg, removes the poisoning class entirely). Legs: the two
cold-cache AUTO legs (the real #47547 behavior — do mixed
FA2+FLASHINFER engines actually work?), the missing 30K FLASHINFER
no-spec, and the 50K quartet.

**Requirements**: A100 80G runtime, `HF_TOKEN` in Colab Secrets. Warm
runtime: ~2-2.5h GPU. Fresh runtime: + install + 22 GiB model download.
Run strictly top to bottom. Full engine logs land in `legN-*.log`.

**Send back**: each leg cell's output + the final summary cell.


In [ ]:
import subprocess
q = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader,nounits'],
                   capture_output=True, text=True).stdout.strip()
print(q)
name, mem = [s.strip() for s in q.split(',')]
assert 'A100' in name, f'Need an A100 runtime, got: {name}'
assert int(mem) > 60000, (
    f'This notebook needs the 80G A100, got {mem} MiB. The 40G card runs out of '
    'KV cache at these context lengths: Runtime -> Change runtime type.')
print('GPU check OK')


In [ ]:
import subprocess, sys, importlib

def ver():
    try:
        import vllm
        return vllm.__version__
    except ImportError:
        return None

if ver() != '0.28.0':
    print('installing vllm==0.28.0 (fresh runtime; pip resolver ERROR wall is Colab noise)')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'vllm==0.28.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchaudio'], check=False)
    importlib.invalidate_caches()
    assert ver() == '0.28.0', 'restart the runtime, then rerun from the top'

import vllm, flashinfer
print('vllm', vllm.__version__, '| flashinfer', flashinfer.__version__)


In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF token set')


In [ ]:
# Ensure config.py is in the #47547 state (forced fallback absent).
# Needed only by the AUTO legs; explicit-backend legs ignore it.
# Deterministic from any of the three states a session can leave behind.
import pathlib
import vllm.model_executor.models.config as C

p = pathlib.Path(C.__file__)
src = p.read_text()

MONKEY_NEW = """        elif vllm_config.attention_config.backend is None:
            from vllm.platforms import current_platform
            from vllm.utils.flashinfer import has_flashinfer

            if current_platform.is_cuda() and max_head_dim <= 512 and has_flashinfer():
                vllm_config.attention_config.backend = AttentionBackendEnum.FLASHINFER
                logger.info("Gemma4: FA4 not available; using FLASHINFER (patched).")
            else:
                vllm_config.attention_config.backend = AttentionBackendEnum.TRITON_ATTN
"""
MONKEY_OLD = """        elif vllm_config.attention_config.backend is None:
            vllm_config.attention_config.backend = AttentionBackendEnum.TRITON_ATTN
"""
OLD_BLOCK = """        elif vllm_config.attention_config.backend is None:
            vllm_config.attention_config.backend = AttentionBackendEnum.TRITON_ATTN
            logger.info(
                "Gemma4 model has heterogeneous head dimensions "
                "%s. FA4 not available, forcing TRITON_ATTN backend.",
                head_dims,
            )
"""
if MONKEY_NEW in src:
    src = src.replace(MONKEY_NEW, MONKEY_OLD)
if OLD_BLOCK in src:
    src = src.replace(OLD_BLOCK, "")
assert 'forcing TRITON_ATTN backend' not in src and '(patched)' not in src, \
    'config.py in unrecognized state; pip install --force-reinstall --no-deps vllm==0.28.0'
p.write_text(src)
print('config.py in #47547 state (selector decides when no backend is set)')


In [ ]:
%%writefile probe_matrix.py
import sys, time
from vllm import LLM, SamplingParams

backend = sys.argv[1]            # FLASHINFER | TRITON_ATTN | AUTO
spec    = sys.argv[2] == "mtp"   # mtp | nospec
ctx     = int(sys.argv[3])       # 30000 | 50000

MODEL = "google/gemma-4-31B-it-qat-w4a16-ct"
ASSIST = "google/gemma-4-31B-it-assistant"

kwargs = dict(model=MODEL,
              max_model_len=33000 if ctx == 30000 else ctx + 1000,
              gpu_memory_utilization=0.9,
              limit_mm_per_prompt={"image": 0, "video": 0, "audio": 0})
if backend != "AUTO":
    kwargs["attention_backend"] = backend
if spec:
    kwargs["speculative_config"] = {"method": "mtp", "model": ASSIST,
                                    "num_speculative_tokens": 1}

llm = LLM(**kwargs)
tok = llm.get_tokenizer()
ids = tok("hello " * (ctx + 20000)).input_ids[:ctx]
prompt = tok.decode(ids)

def timed(n):
    sp = SamplingParams(temperature=0, max_tokens=n, ignore_eos=True)
    t0 = time.time(); llm.generate([prompt], sp); return time.time() - t0

timed(8)
t8, t64 = timed(8), timed(64)
tps = (64 - 8) / (t64 - t8)
print(f"RESULT decode_tok_s={tps:.2f}  t8={t8:.1f}s t64={t64:.1f}s", flush=True)


In [ ]:
# Runner v3: clears the compile cache before EVERY leg (configs never
# repeat, so caching buys nothing and stale-artifact poisoning is fatal),
# always prints exit code + evidence lines, dumps the ERROR tail on
# failure, saves the full log to disk.
import subprocess, sys, pathlib, shutil

CACHE = "/root/.cache/vllm/torch_compile_cache"
KEYS = ("heterogeneous head dimensions", "RESULT", "FLASHINFER", "TRITON_ATTN",
        "KV cache layout", "Disabled mm_prefix", "text-only mode", "AOT compil",
        "not valid for this configuration", "Engine core initialization failed")

def run_probe(script, tag, args=()):
    shutil.rmtree(CACHE, ignore_errors=True)
    r = subprocess.run([sys.executable, script, *args], capture_output=True, text=True)
    out = r.stdout + r.stderr
    pathlib.Path(tag + '.log').write_text(out)
    print(f"=== {tag}: exit code {r.returncode} | full log: {tag}.log ===")
    hits = [l for l in out.splitlines() if any(k in l for k in KEYS)]
    for l in hits: print(l)
    if not hits: print("(no evidence lines matched)")
    if r.returncode != 0:
        errs = [l for l in out.splitlines() if "ERROR" in l]
        print(f"--- last 25 ERROR lines of {tag} ---")
        print("\n".join(errs[-25:]))
    return out

print('runner v3 ready (cold cache per leg)')


## Legs A/B — the discriminator: AUTO selection on a cold cache

This is what #47547 actually produces on Ampere text-only (mixed
FA2 sliding + FLASHINFER full-512). 3b/4b crashed with a poisoned
cache; cold-cache runs answer whether mixed-backend engines work at
all. If a leg still crashes, the printed ERROR tail is the finding.


In [ ]:
outA = run_probe("probe_matrix.py", "leg4d-auto-mixed-nospec-coldcache", ("AUTO", "nospec", "30000"))


In [ ]:
outB = run_probe("probe_matrix.py", "leg3d-auto-mixed-mtp-coldcache", ("AUTO", "mtp", "30000"))


## Legs C-G — explicit-backend measurements (state-independent)

30K FLASHINFER no-spec completes the 30K 2x2; then the 50K quartet
answers whether the TRITON collapse deepens toward the reporter's 3x at
their depth, and what FLASHINFER delivers there.


In [ ]:
outC = run_probe("probe_matrix.py", "leg4c-flashinfer-nospec", ("FLASHINFER", "nospec", "30000"))


In [ ]:
outD = run_probe("probe_matrix.py", "leg5c-triton-mtp-50k", ("TRITON_ATTN", "mtp", "50000"))


In [ ]:
outE = run_probe("probe_matrix.py", "leg6c-triton-nospec-50k", ("TRITON_ATTN", "nospec", "50000"))


In [ ]:
outF = run_probe("probe_matrix.py", "leg7c-flashinfer-mtp-50k", ("FLASHINFER", "mtp", "50000"))


In [ ]:
outG = run_probe("probe_matrix.py", "leg8c-flashinfer-nospec-50k", ("FLASHINFER", "nospec", "50000"))


## Summary — paste this cell's output back


In [ ]:
import pathlib, re

MAIN = [("leg1-triton-mtp",                 "30K TRITON + MTP (default)"),
        ("leg2-triton-nospec",              "30K TRITON, no spec"),
        ("leg3c-flashinfer-explicit-mtp",   "30K FLASHINFER + MTP"),
        ("leg4c-flashinfer-nospec",         "30K FLASHINFER, no spec"),
        ("leg3d-auto-mixed-mtp-coldcache",  "30K AUTO mixed + MTP, cold"),
        ("leg4d-auto-mixed-nospec-coldcache", "30K AUTO mixed no spec, cold"),
        ("leg5c-triton-mtp-50k",            "50K TRITON + MTP"),
        ("leg6c-triton-nospec-50k",         "50K TRITON, no spec"),
        ("leg7c-flashinfer-mtp-50k",        "50K FLASHINFER + MTP"),
        ("leg8c-flashinfer-nospec-50k",     "50K FLASHINFER, no spec")]
DEAD = [("leg3b-flashinfer-mtp",   "crashed: stale AOT cache"),
        ("leg4b-flashinfer-nospec", "crashed: stale AOT cache"),
        ("leg3-flashinfer-mtp",    "superseded (mm still enabled)"),
        ("leg4-flashinfer-nospec", "superseded (mm still enabled)")]

print(f"{'leg':<34}{'decode tok/s':>14}")
for tag, label in MAIN:
    f = pathlib.Path(tag + '.log')
    if not f.exists():
        print(f"{label:<34}{'(not run)':>14}"); continue
    m = re.search(r"RESULT decode_tok_s=([\d.]+)\s+t8=([\d.]+)s t64=([\d.]+)s", f.read_text())
    if m:
        print(f"{label:<34}{m.group(1):>14}   t8={m.group(2)}s t64={m.group(3)}s")
    else:
        print(f"{label:<34}{'(crashed)':>14}")
for tag, note in DEAD:
    if pathlib.Path(tag + '.log').exists():
        print(f"{tag:<34}{'-':>14}   {note}")

print()
for tag, label in MAIN:
    f = pathlib.Path(tag + '.log')
    if not f.exists(): continue
    print(f"--- {tag} ---")
    seen = set()
    for l in f.read_text().splitlines():
        if any(k in l for k in ("Using AttentionBackendEnum", "out of potential backends",
                                "Disabled mm_prefix", "text-only mode",
                                "torch_aot_compile", "not valid for this configuration")):
            key = l.split("] ", 1)[-1]
            if key not in seen:
                seen.add(key); print(l)
